# Instance-level visual search cascade

Experimental notebook: orchestration and result tables only. All logic lives in the `visual_search` package.

**Status:** step 5. Working so far: datasets, the evaluation harness, baselines, **DINOv2 stage-1 retrieval** and the **interactive prompting UI** (box masks for now; SAM3 once it passes its manual check). Matching and re-ranking come in later steps; see `docs/log/`.

The interactive app is at the bottom. It needs JupyterLab with ipympl (`uv run jupyter lab`).

In [ ]:
%matplotlib widget
import logging
from pathlib import Path

import pandas as pd

from visual_search.cascade import Cascade
from visual_search.data import load_folder_dataset, make_synthetic_dataset, parse_dataset, split_dataset, validate_dataset
from visual_search.device import device_sync
from visual_search.eval import comparison_table, evaluate
from visual_search.masking import BoxMasker, NoMasker
from visual_search.models.dinov2 import Dinov2Backbone
from visual_search.retrieval import POOLINGS, ColorHistogramRetriever, DinoRetriever, RandomRetriever
from visual_search.ui import SearchApp

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "cache"  # on-disk feature cache (gitignored)
SEED = 0

# Stage-1 configuration. Any facebook/dinov2-* or facebook/dinov2-with-registers-* checkpoint is a drop-in.
DINO_MODEL = "facebook/dinov2-small"
DINO_RESOLUTION = 448  # multiple of 14; 224 is ~3x faster, a bit less accurate
RETRIEVAL_K = 100

## Datasets
- **Synthetic** (SMOKE TEST ONLY): procedural textures in cluttered scenes. Useful to check that everything runs, not to compare methods.
- **Real**: `data/<instance_id>/*.jpg`, with distractors in `data/singletons/`. See the README.

In [ ]:
synthetic_split = split_dataset(
    make_synthetic_dataset(n_instances=12, views_per_instance=3, n_distractors=40, n_near_duplicates=3, seed=SEED),
    seed=SEED,
)
print("synthetic:", synthetic_split.summary())

real_records = load_folder_dataset(DATA_DIR) if DATA_DIR.is_dir() else []
print(validate_dataset(parse_dataset(real_records)).summary())
real_split = split_dataset(real_records, seed=SEED, strict=False) if real_records else None
print("real:", real_split.summary() if real_split else "no data")

splits = {"synthetic": synthetic_split}
if real_split and real_split.queries:
    splits["real"] = real_split

## Stage 1: baselines vs DINOv2 pooling modes
- `cls`: global CLS token of the full query image (ignores the prompt).
- `crop_cls`: CLS of the query cropped to its prompt box (one extra forward pass per query).
- `masked_mean`: mean of the patch tokens under the prompt (no extra forward pass, so prompt changes are instant).

Gallery images are described as whole images. Features are cached in `cache/`, so re-runs only pay for the queries.
Latency is per query in ms (median / p95). `prepare_s` is the offline gallery-indexing time.

In [ ]:
backbone = Dinov2Backbone(DINO_MODEL, resolution=DINO_RESOLUTION)
sync = device_sync(backbone.device)
print("device:", backbone.device, "| dim:", backbone.dim, "| patch grid:", backbone.grid)


def stage1_reports(split):
    """Evaluate the baselines and every DINOv2 pooling mode on a split."""
    reports = [evaluate(Cascade(RandomRetriever(seed=SEED)), split), evaluate(Cascade(ColorHistogramRetriever()), split)]
    for pooling in POOLINGS:
        retriever = DinoRetriever(backbone, pooling=pooling, cache_dir=CACHE_DIR)
        reports.append(evaluate(Cascade(retriever, retrieval_k=RETRIEVAL_K, sync=sync), split))
    return reports


reports = {name: stage1_reports(split) for name, split in splits.items()}
tables = {}
for name, reps in reports.items():
    t = comparison_table(reps).drop(columns=["stage"])
    t["prepare_s"] = [sum(r.prepare_s.values()) for r in reps]
    tables[name] = t
pd.concat(tables, names=["dataset"]).round(3)

## Per-query inspection (DINOv2 `crop_cls` on synthetic)
Rank of the first correct gallery item and the top-1 result per query; large ranks are the failures to look at.

In [ ]:
best = next(r for r in reports["synthetic"] if r.name.startswith("dinov2-small/crop_cls"))
best.per_query.sort_values("rank@retrieval", ascending=False)

## Interactive search
1. Pick a query from the dropdown, or type an image path and press **Load path**.
2. Draw a box (**Box** mode) or click points (**Points** mode: left click = positive, right click or shift+click = negative). The mask preview updates after every change.
3. Press **Search**. You get the top-k per stage with scores and labels, ✓/✗ when the query is labelled, and per-step timings.

The masker is a box mask for now. Once SAM3 fp16 passes `scripts/try_sam3.py` on your machine, switch to:
```python
from visual_search.masking.sam3 import Sam3Config, Sam3Masker
MASKER = Sam3Masker(Sam3Config(model_dir=ROOT / "models/sam3", variant="fp16"))
```

In [ ]:
APP_SPLIT = splits.get("real", synthetic_split)  # your data if it has queries, else the synthetic set
MASKER = BoxMasker()  # NoMasker() = ablation without masks
app_retriever = DinoRetriever(backbone, pooling="crop_cls", cache_dir=CACHE_DIR)
app = SearchApp(
    Cascade(app_retriever, retrieval_k=RETRIEVAL_K, sync=sync),
    gallery=APP_SPLIT.gallery,
    queries=APP_SPLIT.queries,
    masker=MASKER,
    top_k=5,
)
app.widget

After a search, `app.last_result` holds the full rankings per stage and `app.timings` the per-step latency.